# 01 â€” CelebA Data Preparation

This notebook walks from CelebA's aligned source crops to the project datasets: first the single-identity classification split, then the multi-identity synthetic frames with face-level YOLO labels and label-aware augmentation. It records source-split counts, checks for leakage and class coverage, and displays annotated frame samples inline.

Prerequisites: place `img_align_celeba/` and `identity_CelebA.txt` under `data/`, install `requirements.txt`, and run this notebook with the repository's `.venv` kernel.

In [ ]:
import sys, json
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dataset import build_splits, CelebASubset, IMAGE_SIZE
import pandas as pd
import matplotlib.pyplot as plt

LOGS_DIR = REPO_ROOT / "logs"

## Load the selected identities

Produced by `python src/select_identities.py` (see that script for the selection rationale).

In [ ]:
selected_path = LOGS_DIR / "selected_identities.json"
assert selected_path.exists(), "Run `python src/select_identities.py --identity-ids 3 7 1212 8335` first."

selected = json.loads(selected_path.read_text())
IDENTITY_IDS = [row["identity_id"] for row in selected["selected"]]
print("Using identities:", IDENTITY_IDS)
for row in selected["selected"]:
    print(f"  id={row['identity_id']:>6}  n_images={row['num_images']:>3}")

## Build the stratified train/val/test split (15% val, 15% test, per identity)

In [ ]:
splits_df = build_splits(IDENTITY_IDS, val_frac=0.15, test_frac=0.15, seed=42)
print(splits_df.groupby(["identity", "split"]).size().unstack(fill_value=0))


## Sanity-check a batch of preprocessed/augmented training images

In [ ]:
import torch
from torch.utils.data import DataLoader

train_ds = CelebASubset("train", splits_df)
val_ds = CelebASubset("val", splits_df)
test_ds = CelebASubset("test", splits_df)
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}  num_classes={train_ds.num_classes}")

loader = DataLoader(train_ds, batch_size=8, shuffle=True)
images, labels = next(iter(loader))

# undo normalization for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for img, label, ax in zip(images, labels, axes.flatten()):
    img = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(f"identity label {label.item()}")
    ax.axis("off")
fig.suptitle(f"Sample augmented training batch ({IMAGE_SIZE}x{IMAGE_SIZE})")
fig.tight_layout()
fig.savefig(LOGS_DIR / "sample_batch.png", dpi=150)
plt.show()


## Classification baseline preprocessing

The cells above create the single-face classification dataset used by the CNN and ResNet18 notebooks:

- **Resize**: all images are resized to 128x128.
- **Normalization**: ImageNet mean/std is applied consistently to both models.
- **Train-only augmentation**: horizontal flip and light color jitter; validation and test images are not augmented.
- **Split**: stratified by identity for the classification baseline.

The synthetic detection pipeline below uses separate 640x640 frames, source-disjoint 70/15/15 splits, YOLO face boxes, and box-aware frame augmentation.

## Raw CelebA crops and identity mapping

The source archive contains aligned 178x218 JPEG crops. `identity_CelebA.txt` maps each image filename to an identity ID; the four selected IDs have 99 source crops total (25, 24, 25, and 25). The next cell audits the files and renders one raw crop for each identity before composition.

In [ ]:
from PIL import Image

RAW_SOURCE_DIR = REPO_ROOT / "data" / "img_align_celeba"
IDENTITY_MAP_PATH = REPO_ROOT / "data" / "identity_CelebA.txt"
identity_map = pd.read_csv(
    IDENTITY_MAP_PATH, sep=r"\s+", header=None, names=["image_id", "identity_id"]
)
raw_counts = identity_map[identity_map["identity_id"].isin(IDENTITY_IDS)].groupby("identity_id").size()
raw_counts = raw_counts.reindex(IDENTITY_IDS, fill_value=0)
print("Raw CelebA source crops by identity:")
print(raw_counts)
assert (raw_counts > 0).all()
assert all(
    (RAW_SOURCE_DIR / image_name).is_file()
    for image_name in identity_map.loc[identity_map["identity_id"].isin(IDENTITY_IDS), "image_id"]
)

fig, axes = plt.subplots(1, len(IDENTITY_IDS), figsize=(12, 4))
for identity_id, ax in zip(IDENTITY_IDS, axes):
    image_name = identity_map.loc[
        identity_map["identity_id"] == identity_id, "image_id"
    ].iloc[0]
    with Image.open(RAW_SOURCE_DIR / image_name) as source_image:
        ax.imshow(source_image.convert("RGB"))
    ax.set_title(f"Identity {identity_id} | {raw_counts[identity_id]} crops")
    ax.axis("off")
fig.suptitle("Raw aligned CelebA crops")
fig.tight_layout()
plt.show()

## Compose, detect, augment, and export YOLO data

`src/build_synthetic_frames.py` reads the raw aligned crops and identity map, partitions source photos by identity at 70/15/15 **before** composition, then uses YuNet to find each face. It composes 2â€“4 distinct identities over varied classroom-style backgrounds. This order prevents the same source photo from appearing in multiple dataset splits.

| Augmentation | Parameters | Why it is appropriate for detection |
|---|---|---|
| Per-portrait horizontal mirror | Probability 0.5 | Adds left/right pose variation; the source face box is mirrored with the portrait. |
| Per-portrait appearance jitter | Brightness 0.92â€“1.08, contrast 0.95â€“1.05, saturation 0.94â€“1.06 | Adds modest subject lighting/color variation without removing face detail. |
| Whole-frame horizontal flip | Probability 0.5 | Varies group composition; all face boxes flip with the frame. |
| Whole-frame scale | 0.92â€“1.08, centered | Simulates framing distance; applies the same resize to every box. |
| Whole-frame rotation | -6Â° to +6Â° | Simulates mild camera tilt; transforms all four box corners, then recomputes an axis-aligned box. |
| Whole-frame brightness/contrast | Each factor 0.90â€“1.10 | Simulates exposure and contrast changes without affecting labels. |

Boxes are clipped to frame bounds after geometric transforms. Each sampled frame-level parameter is recorded in `manifest.csv`. The output has YOLO `images/{train,val,test}` and matching `labels/{train,val,test}`, plus `dataset.yaml`, source-split records, and review overlays. The next cell reuses the local dataset if present; otherwise it runs the generator.

In [ ]:
import subprocess

SYNTHETIC_DATA_DIR = REPO_ROOT / "data" / "synthetic_face_frames_70_15_15"
if not (SYNTHETIC_DATA_DIR / "dataset.yaml").exists():
    subprocess.run(
        [sys.executable, str(REPO_ROOT / "src" / "build_synthetic_frames.py")],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print(f"Reusing generated dataset at {SYNTHETIC_DATA_DIR}")

from src.build_synthetic_frames import write_visual_checks
write_visual_checks(SYNTHETIC_DATA_DIR, IDENTITY_IDS)
print("YOLO dataset config:", SYNTHETIC_DATA_DIR / "dataset.yaml")
print("Label directories:", *(SYNTHETIC_DATA_DIR / "labels" / split for split in ("train", "val", "test")), sep="\n  ")

## Synthetic-frame split rationale and counts

We split the original CelebA source photos by identity **before** composition using a 70/15/15 train/validation/test target and a seeded largest-remainder allocation. This ensures each source image belongs to exactly one split, so the same photo cannot leak into another split through a different composite. For the 23â€“25 source photos per identity, the exact integer counts are:

| Identity ID | Source images | Train | Validation | Test |
|---:|---:|---:|---:|---:|
| 3 | 25 | 17 | 4 | 4 |
| 7 | 24 | 17 | 4 | 3 |
| 1212 | 25 | 17 | 4 | 4 |
| 8335 | 25 | 17 | 4 | 4 |
| **Total** | **99** | **68** | **16** | **15** |

The seeded frame schedule guarantees every identity appears in every split when a split has at least one frame. Each frame contains 2â€“4 distinct identities, so per-identity frame counts vary:

| Identity ID | Train frames | Validation frames | Test frames |
|---:|---:|---:|---:|
| 3 | 152 | 30 | 34 |
| 7 | 156 | 34 | 28 |
| 1212 | 144 | 35 | 28 |
| 8335 | 151 | 28 | 33 |
| **Total frames** | **200** | **40** | **40** |

The following cell reads the generated manifests to reproduce these counts and assert source-image isolation and identity coverage.

In [ ]:
SYNTHETIC_DATA_DIR = REPO_ROOT / "data" / "synthetic_face_frames_70_15_15"
SPLIT_ORDER = ["train", "val", "test"]

source_splits = pd.read_csv(SYNTHETIC_DATA_DIR / "source_splits.csv")
source_counts = pd.crosstab(
    source_splits["identity_id"], source_splits["split"]
).reindex(columns=SPLIT_ORDER, fill_value=0)
print("Source-image counts by identity:")
display(source_counts)
print("Source-image totals:", source_counts.sum().to_dict())
assert source_splits.groupby("source_image")["split"].nunique().max() == 1
assert (source_counts > 0).all().all(), "A source identity is missing from a split."

frame_manifest = pd.read_csv(SYNTHETIC_DATA_DIR / "manifest.csv")
frame_counts = pd.crosstab(
    frame_manifest["identity_id"], frame_manifest["split"],
    values=frame_manifest["image"], aggfunc="nunique",
).fillna(0).astype(int).reindex(columns=SPLIT_ORDER, fill_value=0)
print("Distinct synthetic frames containing each identity:")
display(frame_counts)
frame_totals = frame_manifest.groupby("split")["image"].nunique().reindex(SPLIT_ORDER)
print("Synthetic-frame totals:", frame_totals.to_dict())
expected_frame_counts = pd.DataFrame.from_dict(
    {
        3: {"train": 152, "val": 30, "test": 34},
        7: {"train": 156, "val": 34, "test": 28},
        1212: {"train": 144, "val": 35, "test": 28},
        8335: {"train": 151, "val": 28, "test": 33},
    },
    orient="index",
).reindex(index=frame_counts.index, columns=SPLIT_ORDER)
assert frame_counts.equals(expected_frame_counts), "Frame counts differ from the documented seed-42 run."
assert (frame_counts > 0).all().all(), "A synthetic-frame identity is missing from a split."
assert sum(frame_totals) == 280
print("Verified: source photos are split-disjoint and every identity occurs in every frame split.")

## Inspect the YOLO labels

Each text file beside an image has one row per detected face: `class_id x_center y_center width height`. The four class IDs map to identities 3, 7, 1212, and 8335. Coordinates are normalized to the 640x640 frame, and the following audit checks image/label pairing, 2â€“4 faces per frame, valid classes, and in-bounds boxes.

In [ ]:
CLASS_TO_IDENTITY = {0: 3, 1: 7, 2: 1212, 3: 8335}
label_row_counts = {}
for split in SPLIT_ORDER:
    image_paths = sorted((SYNTHETIC_DATA_DIR / "images" / split).glob("*.jpg"))
    label_paths = sorted((SYNTHETIC_DATA_DIR / "labels" / split).glob("*.txt"))
    assert len(image_paths) == len(label_paths)
    for image_path in image_paths:
        label_path = SYNTHETIC_DATA_DIR / "labels" / split / f"{image_path.stem}.txt"
        assert label_path.is_file(), f"Missing label file for {image_path.name}"
        rows = [line.split() for line in label_path.read_text().splitlines()]
        assert 2 <= len(rows) <= 4, f"Unexpected face count in {label_path.name}"
        seen_classes = set()
        for row in rows:
            assert len(row) == 5, f"Expected five YOLO fields in {label_path.name}"
            class_id, center_x, center_y, width, height = row
            class_id = int(class_id)
            center_x, center_y, width, height = map(float, (center_x, center_y, width, height))
            assert class_id in CLASS_TO_IDENTITY
            assert class_id not in seen_classes, f"Duplicate identity in {label_path.name}"
            seen_classes.add(class_id)
            assert 0 <= center_x - width / 2 < center_x + width / 2 <= 1
            assert 0 <= center_y - height / 2 < center_y + height / 2 <= 1
    label_row_counts[split] = len(label_paths)
    print(f"{split}: {len(image_paths)} images, {len(label_paths)} matching YOLO labels; boxes valid")

print("Class mapping:", CLASS_TO_IDENTITY)
example_label = sorted((SYNTHETIC_DATA_DIR / "labels" / "train").glob("*.txt"))[0]
print(f"Example ({example_label.name}):")
print(example_label.read_text())

## Inline visual check of annotated synthetic frames

Red rectangles show the normalized YuNet face boxes reconstructed from each YOLO label. Two examples from each split are embedded below; the generator recreates these review images from the frame/label pairs.

### Train

![Train frame with face boxes](../data/synthetic_face_frames_70_15_15/visual_checks/train_box_review_00000.jpg)

![Second train frame with face boxes](../data/synthetic_face_frames_70_15_15/visual_checks/train_box_review_00100.jpg)

### Validation

![Validation frame with face boxes](../data/synthetic_face_frames_70_15_15/visual_checks/val_box_review_00000.jpg)

![Second validation frame with face boxes](../data/synthetic_face_frames_70_15_15/visual_checks/val_box_review_00020.jpg)

### Test

![Test frame with face boxes](../data/synthetic_face_frames_70_15_15/visual_checks/test_box_review_00000.jpg)

![Second test frame with face boxes](../data/synthetic_face_frames_70_15_15/visual_checks/test_box_review_00020.jpg)